In [1]:
%run "/Users/manojrammopati/Public/Projects/bupa_insurance_project/_01_Bronze/Jupyter Notebooks/00_bronze_data_connector.ipynb"


25/12/17 17:37:03 WARN Utils: Your hostname, Manojrams-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 192.168.0.219 instead (on interface en0)
25/12/17 17:37:03 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/manojrammopati/.ivy2/cache
The jars for the packages stored in: /Users/manojrammopati/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-azure added as a dependency
com.azure#azure-storage-blob added as a dependency
com.azure#azure-identity added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-af893c5b-d8cb-4583-aeb5-58c0504c09fb;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.1.0 in central


:: loading settings :: url = jar:file:/opt/anaconda3/envs/spark_local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found io.delta#delta-storage;3.1.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-azure;3.3.4 in central
	found org.apache.httpcomponents#httpclient;4.5.13 in central
	found org.apache.httpcomponents#httpcore;4.4.13 in central
	found commons-logging#commons-logging;1.1.3 in central
	found commons-codec#commons-codec;1.15 in central
	found com.microsoft.azure#azure-storage;7.0.1 in central
	found com.fasterxml.jackson.core#jackson-core;2.12.7 in central
	found org.slf4j#slf4j-api;1.7.36 in central
	found com.microsoft.azure#azure-keyvault-core;1.0.0 in central
	found com.google.guava#guava;27.0-jre in central
	found com.google.guava#failureaccess;1.0 in central
	found com.google.guava#listenablefuture;9999.0-empty-to-avoid-conflict-with-guava in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found org.checkerframework#checker-qual;2.5.2 in central
	found com.google.errorprone#error_prone_annotations;2.2.0 in central
	found 

Preconnectors ran successfully and connected to 'clientdatastorage' storage accounts
Created 33 tables in bupa_gold database


In [2]:
# Cell 0 — Optional: ensure Spark & paths helpers are available
# (same pattern you use in other notebooks)

import sys
import importlib
import os

# Adjust PROJECT_ROOT if needed
PROJECT_ROOT = "/Users/manojrammopati/Public/Projects/bupa_insurance_project"

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Add Silver utils into path
silver_dir = os.path.join(PROJECT_ROOT, "_02_Silver")
if silver_dir not in sys.path:
    sys.path.insert(0, silver_dir)

import utils_silver
importlib.reload(utils_silver)

print("Loaded utils_silver from:", utils_silver.__file__)


✅ utils_silver.py loaded
✅ utils_silver.py loaded
Loaded utils_silver from: /Users/manojrammopati/Public/Projects/bupa_insurance_project/_02_Silver/utils_silver.py


# 🔹 Cell 1 – Imports & config

In [5]:
# Cell 1 — Imports & configuration

from pyspark.sql import functions as F
from pyspark.sql import DataFrame
from pyspark.ml import PipelineModel
from pyspark.ml.functions import vector_to_array

import sys, importlib

# ------------------------------------------------------------------
# Storage configuration (same account/containers as the rest of project)
# ------------------------------------------------------------------
STORAGE_ACCOUNT  = "clientdatastorage"
GOLD_CONTAINER   = "golddata"
DB_GOLD          = "bupa_gold"

# Feature table used for scoring (no split, no label)
FT_POLICY_CHURN_PATH = (
    f"abfss://{GOLD_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/ft_policy_churn"
)

# Where the trained best model was saved (from 01_policy_churn_training.ipynb)
MODEL_BASE_PATH = (
    f"abfss://{GOLD_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/models/policy_churn"
)
MODEL_NAME = "randomforest_v1"
MODEL_PATH = f"{MODEL_BASE_PATH}/{MODEL_NAME}"

# Where we will store batch scoring output
SCORED_PATH = (
    f"abfss://{GOLD_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/scored_policy_churn"
)

print("Feature table path:", FT_POLICY_CHURN_PATH)
print("Model path        :", MODEL_PATH)
print("Scored output path:", SCORED_PATH)


Feature table path: abfss://golddata@clientdatastorage.dfs.core.windows.net/ft_policy_churn
Model path        : abfss://golddata@clientdatastorage.dfs.core.windows.net/models/policy_churn/randomforest_v1
Scored output path: abfss://golddata@clientdatastorage.dfs.core.windows.net/scored_policy_churn


# 🔹 Cell 2 – Load features to be scored

In [6]:
# Cell 2 — Load ft_policy_churn and basic checks

features_all = (
    spark.read
         .format("delta")
         .load(FT_POLICY_CHURN_PATH)
)

print("Total rows in ft_policy_churn:", features_all.count())
features_all.printSchema()
features_all.show(5, truncate=False)

# Optional: keep only data-quality-valid rows, like in training
features_scoring = features_all.filter(
    (F.col("dq_money_valid") == 1) &
    (F.col("dq_discount_valid") == 1) &
    (F.col("dq_renewal_valid") == 1)
)

print("Rows after dq filtering:", features_scoring.count())


25/12/17 17:39:58 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Total rows in ft_policy_churn: 381109
root
 |-- Policy_ID: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Product_Line: string (nullable = true)
 |-- Channel: string (nullable = true)
 |-- Sum_Insured_GBP: double (nullable = true)
 |-- Annual_Premium_GBP: double (nullable = true)
 |-- Policy_Start_Date: date (nullable = true)
 |-- Policy_End_Date: date (nullable = true)
 |-- Policy_Duration_Days: integer (nullable = true)
 |-- Renewal_Offered_Flag: integer (nullable = true)
 |-- Renewal_Accepted_Flag: integer (nullable = true)
 |-- Renewal_Conversion: integer (nullable = true)
 |-- Tenure_Band: string (nullable = true)
 |-- Premium_Band: string (nullable = true)
 |-- Discount_Band: string (nullable = true)
 |-- Renewal_Outcome: string (nullable = true)
 |-- dq_money_valid: integer (nullable = true)
 |-- dq_discount_valid: integer (nullable = true)
 |-- dq_renewal_valid: integer (nullable = true)
 |-- Is_Renewal_Offered: integer (nullable = true)
 |-- Churn_Lab

+----------+------------+------------+-------+------------------+------------------+-----------------+---------------+--------------------+--------------------+---------------------+------------------+-----------+-----------------+-------------+---------------+--------------+-----------------+----------------+------------------+-----------+-------------+-------------------------+
|Policy_ID |Customer_ID |Product_Line|Channel|Sum_Insured_GBP   |Annual_Premium_GBP|Policy_Start_Date|Policy_End_Date|Policy_Duration_Days|Renewal_Offered_Flag|Renewal_Accepted_Flag|Renewal_Conversion|Tenure_Band|Premium_Band     |Discount_Band|Renewal_Outcome|dq_money_valid|dq_discount_valid|dq_renewal_valid|Is_Renewal_Offered|Churn_Label|Is_Discounted|Premium_per_1k_SumInsured|
+----------+------------+------------+-------+------------------+------------------+-----------------+---------------+--------------------+--------------------+---------------------+------------------+-----------+-----------------+---

Rows after dq filtering: 314128


# 🔹 Cell 3 – Define features & null-handling (same logic as training)

This should mirror the logic from 01_policy_churn_training.ipynb as closely as possible.

In [7]:
# Cell 3 — Define feature columns & null-handling

cols = features_scoring.columns

# Numeric features (only keep those that actually exist in the table)
numeric_candidates = [
    "Sum_Insured_GBP",
    "Annual_Premium_GBP",
    "Policy_Duration_Days",
    "Premium_per_1k_SumInsured",    # only if present
]

numeric_cols = [c for c in numeric_candidates if c in cols]

# Categorical features
categorical_candidates = [
    "Product_Line",
    "Channel",
    "Tenure_Band",
    "Premium_Band",
    "Discount_Band",
    "Renewal_Outcome",
]

cat_cols = [c for c in categorical_candidates if c in cols]

print("Numeric features   :", numeric_cols)
print("Categorical features:", cat_cols)

# ID / context columns to keep in the scored output
id_cols = [c for c in [
    "Policy_ID",
    "Customer_ID",
    "Product_Line",
    "Channel"
] if c in cols]

def prep_nulls(df: DataFrame) -> DataFrame:
    """
    Simple null-handling: numeric -> 0, categorical -> 'Unknown'.
    Must match the logic used in the training notebook, so that
    the model "sees" the same distributions.
    """
    d = df
    for c in numeric_cols:
        d = d.withColumn(c, F.when(F.col(c).isNull(), F.lit(0.0)).otherwise(F.col(c)))
    for c in cat_cols:
        d = d.withColumn(c, F.when(F.col(c).isNull(), F.lit("Unknown")).otherwise(F.col(c)))
    return d


Numeric features   : ['Sum_Insured_GBP', 'Annual_Premium_GBP', 'Policy_Duration_Days', 'Premium_per_1k_SumInsured']
Categorical features: ['Product_Line', 'Channel', 'Tenure_Band', 'Premium_Band', 'Discount_Band', 'Renewal_Outcome']


# 🔹 Cell 4 – Load trained model & score all rows

The saved PipelineModel already includes indexers, encoders, assembler, and classifier.
We just need to handle nulls and call .transform.

In [8]:
# Cell 4 — Load best model & score

print("Loading best policy churn model from:", MODEL_PATH)
loaded_model = PipelineModel.load(MODEL_PATH)

# Apply the same null-handling as training
features_prepped = prep_nulls(features_scoring)

scored_raw = loaded_model.transform(features_prepped)

# Probability is a vector; prob[1] = probability of churn (label 1)
scored = (
    scored_raw
        .withColumn("prob_arr", vector_to_array("probability"))
        .withColumn("churn_prob", F.col("prob_arr")[1])
)

# Choose columns for final scored dataset
output_cols = (
    id_cols +
    [
        "Policy_Start_Date", "Policy_End_Date"
    ] if "Policy_Start_Date" in cols and "Policy_End_Date" in cols else id_cols
)

# Make sure the columns exist before selecting
output_cols = [c for c in output_cols if c in scored.columns]

scored_final = scored.select(
    *output_cols,
    F.col("churn_prob"),
    F.col("prediction").alias("churn_prediction")
)

scored_final.printSchema()
scored_final.show(10, truncate=False)

print("Scored rows:", scored_final.count())


Loading best policy churn model from: abfss://golddata@clientdatastorage.dfs.core.windows.net/models/policy_churn/randomforest_v1


root
 |-- Policy_ID: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Product_Line: string (nullable = true)
 |-- Channel: string (nullable = true)
 |-- Policy_Start_Date: date (nullable = true)
 |-- Policy_End_Date: date (nullable = true)
 |-- churn_prob: double (nullable = true)
 |-- churn_prediction: double (nullable = false)



+----------+------------+------------+-------+-----------------+---------------+---------------------+----------------+
|Policy_ID |Customer_ID |Product_Line|Channel|Policy_Start_Date|Policy_End_Date|churn_prob           |churn_prediction|
+----------+------------+------------+-------+-----------------+---------------+---------------------+----------------+
|P_00000001|CUST_0000001|Dental      |Partner|2025-10-12       |2026-05-05     |0.9929039268585131   |1.0             |
|P_00000008|CUST_0000008|Health      |Partner|2021-03-10       |2022-06-08     |1.0                  |1.0             |
|P_00000018|CUST_0000018|Motor       |Partner|2022-01-30       |2023-04-06     |0.0                  |0.0             |
|P_00000031|CUST_0000031|Health      |Partner|2022-01-09       |2023-02-15     |0.0                  |0.0             |
|P_00000035|CUST_0000035|Health      |Partner|2022-04-23       |2023-06-27     |0.0                  |0.0             |
|P_00000049|CUST_0000049|Dental      |Pa

# 🔹 Cell 5 – Write scored output to golddata and register table

In [9]:
DB_GOLD = "bupa_gold"

spark.sql(f"CREATE DATABASE IF NOT EXISTS {DB_GOLD}")

DataFrame[]

In [10]:
# Cell 5 — Persist scored_policy_churn as Delta in golddata and register table

# 1) Write to Delta path in golddata
(
    scored_final
        .write
        .format("delta")
        .mode("overwrite")          # for daily runs you might switch to "append"
        .save(SCORED_PATH)
)

print(f"✅ Wrote scored churn output to: {SCORED_PATH}")

# 2) Register / refresh Hive table in bupa_gold
spark.sql(f"DROP TABLE IF EXISTS {DB_GOLD}.scored_policy_churn")

spark.sql(f"""
    CREATE TABLE {DB_GOLD}.scored_policy_churn
    USING DELTA
    LOCATION '{SCORED_PATH}'
""")

print(f"✅ Registered table: {DB_GOLD}.scored_policy_churn")

# Quick sanity check from SQL
spark.table(f"{DB_GOLD}.scored_policy_churn").show(10, truncate=False)


25/12/17 17:40:50 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


✅ Wrote scored churn output to: abfss://golddata@clientdatastorage.dfs.core.windows.net/scored_policy_churn
✅ Registered table: bupa_gold.scored_policy_churn
+----------+------------+------------+-------+-----------------+---------------+---------------------+----------------+
|Policy_ID |Customer_ID |Product_Line|Channel|Policy_Start_Date|Policy_End_Date|churn_prob           |churn_prediction|
+----------+------------+------------+-------+-----------------+---------------+---------------------+----------------+
|P_00000012|CUST_0000012|Health      |Partner|2025-10-05       |2026-04-05     |0.3761263297280285   |0.0             |
|P_00000020|CUST_0000020|Health      |Partner|2023-10-10       |2024-08-15     |0.0057948792670494   |0.0             |
|P_00000030|CUST_0000030|Dental      |Partner|2025-06-04       |2026-11-17     |0.9930531641745959   |1.0             |
|P_00000032|CUST_0000032|Dental      |Partner|2021-11-10       |2022-09-11     |0.41                 |0.0             |
|P

# 🔹 (Optional) Cell 6 – Lightweight scoring run log

If you’d like to also log the run in ml_monitoring_view (you already have it), you can add this optional cell.
Here we log a scoring run (no AUC/accuracy, just meta info) as a separate “use_case”.

In [11]:
# Cell 6 — OPTIONAL: log scoring run into ml_monitoring_view

# This assumes utils_silver.write_ml_monitoring already exists and works
# and that paths_gold is available via build_paths.

import utils_silver

# Make sure the Silver utils folder is on sys.path (same trick as other notebooks)
silver_dir = "/Users/manojrammopati/Public/Projects/bupa_insurance_project/_02_Silver"
if silver_dir not in sys.path:
    sys.path.insert(0, silver_dir)
importlib.reload(utils_silver)

paths_bronze, paths_silver, paths_gold = utils_silver.build_paths(STORAGE_ACCOUNT)

run_ts = F.current_timestamp().cast("timestamp")
row_count = scored_final.count()
avg_prob  = scored_final.select(F.avg("churn_prob").alias("avg_p")).collect()[0]["avg_p"]

rows_for_log = [{
    "model_name":   "BestPolicyChurnModel",
    "use_case":     "policy_churn_scoring",
    "dataset_name": "ft_policy_churn",
    "dataset_split": "score",
    # No ground truth → AUC/accuracy not meaningful; use 0.0
    "auc":       0.0,
    "accuracy":  0.0,
    "precision": 0.0,
    "recall":    0.0,
    "f1":        0.0,
    "ts":        None,   # utils_silver will create its own timestamp column
    "notes":     f"Batch scoring run — rows={row_count}, avg_churn_prob={avg_prob:.4f}",
}]

utils_silver.write_ml_monitoring_view(
    spark=spark,
    rows=rows_for_log,
    paths_gold=paths_gold,
)

print("✅ Logged scoring run into ml_monitoring_view")


✅ utils_silver.py loaded


AnalysisException: A schema mismatch detected when writing to the Delta table (Table ID: 5d572a7b-26ec-4752-8b4b-0096b3981990).
To enable schema migration using DataFrameWriter or DataStreamWriter, please set:
'.option("mergeSchema", "true")'.
For other operations, set the session configuration
spark.databricks.delta.schema.autoMerge.enabled to "true". See the documentation
specific to the operation for details.

Table schema:
root
-- Claim_ID: string (nullable = true)
-- Member_Key: string (nullable = true)
-- Provider_ID: string (nullable = true)
-- Is_Fraudulent_Claim: integer (nullable = true)
-- fraud_prob: double (nullable = true)
-- prediction: double (nullable = true)
-- Claim_Type_Name: string (nullable = true)
-- Claim_Status: string (nullable = true)
-- Claim_Amount_GBP: double (nullable = true)
-- Payout_GBP: double (nullable = true)
-- Payout_to_Amount_Ratio: double (nullable = true)
-- Provider_Risk_Tier: string (nullable = true)
-- dataset_split: string (nullable = true)